# 02 — HIFLD Transmission Lines & Power Plants

**Purpose:** Load, filter, reproject, and map HIFLD transmission line and 
power plant shapefiles for high-voltage network analysis.

**Inputs (manual download required — see cell below):**
- `data/raw/transmission_lines/*.shp` — HIFLD Electric Power Transmission Lines
- `data/raw/hifld_plants/*.shp` — HIFLD Power Plants

**Outputs:**
- `data/processed/transmission_lines_hv.geojson` — Lines ≥200 kV, EPSG:4326
- `data/processed/hifld_plants.geojson` — HIFLD plants, EPSG:4326
- `data/processed/grid_overview_map.html` — Combined Folium map

## ⚠️ Manual Download Required

Before running this notebook, download the following two datasets from HIFLD:

**Source:** https://hifld-geoplatform.opendata.arcgis.com

1. **Electric Power Transmission Lines**
   - Search: `"Electric Power Transmission Lines"`
   - Download as **Shapefile**
   - Extract contents into: `data/raw/transmission_lines/`

2. **Power Plants**
   - Search: `"Power Plants"`
   - Download as **Shapefile**
   - Extract contents into: `data/raw/hifld_plants/`

The notebook will glob for the `.shp` file inside each directory, so exact 
filenames don't matter as long as they're in the right folder.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import geopandas as gpd
import folium

import utils

## 1. Transmission Lines

In [ ]:
# ── Locate shapefile ───────────────────────────────────────────────────────────
tl_dir = PROJECT_ROOT / "data" / "raw" / "transmission_lines"
shp_files = list(tl_dir.glob("*.shp"))
assert shp_files, f"No .shp file found in {tl_dir}. Did you download and extract the shapefile?"
tl_shp = shp_files[0]
print(f"Found shapefile: {tl_shp}")

In [ ]:
# ── Read transmission lines ────────────────────────────────────────────────────
print("Reading transmission lines shapefile...")
tl = gpd.read_file(tl_shp)

print(f"Shape: {tl.shape}")
print(f"CRS:   {tl.crs}")
print(f"\nColumns: {list(tl.columns)}")
print("\nFirst 3 rows:")
tl.head(3)

In [ ]:
# ── Filter to high-voltage lines (≥200 kV) ─────────────────────────────────────
# HIFLD uses -999 as a null sentinel for VOLTAGE
before = len(tl)
tl_hv = tl[(tl["VOLTAGE"] >= 200) & (tl["VOLTAGE"] != -999)].copy()
after = len(tl_hv)
print(f"Filtered {before:,} → {after:,} lines (VOLTAGE ≥ 200 kV, excluding -999 nulls)")

In [ ]:
# ── Reproject to EPSG:4326 if needed ──────────────────────────────────────────
if tl_hv.crs and tl_hv.crs.to_epsg() != 4326:
    print(f"Reprojecting from {tl_hv.crs} → EPSG:4326...")
    tl_hv = tl_hv.to_crs(epsg=4326)
else:
    print("Already in EPSG:4326.")
print(f"CRS after reproject: {tl_hv.crs}")

In [ ]:
# ── Save filtered transmission lines ──────────────────────────────────────────
utils.save_processed(tl_hv, "transmission_lines_hv.geojson")

## 2. HIFLD Power Plants

In [ ]:
# ── Locate shapefile ───────────────────────────────────────────────────────────
hp_dir = PROJECT_ROOT / "data" / "raw" / "hifld_plants"
hp_shp_files = list(hp_dir.glob("*.shp"))
assert hp_shp_files, f"No .shp file found in {hp_dir}. Did you download and extract the shapefile?"
hp_shp = hp_shp_files[0]
print(f"Found shapefile: {hp_shp}")

In [ ]:
# ── Read HIFLD plants ──────────────────────────────────────────────────────────
print("Reading HIFLD power plants shapefile...")
hp = gpd.read_file(hp_shp)
print(f"Shape: {hp.shape}, CRS: {hp.crs}")

if hp.crs and hp.crs.to_epsg() != 4326:
    print(f"Reprojecting from {hp.crs} → EPSG:4326...")
    hp = hp.to_crs(epsg=4326)

utils.save_processed(hp, "hifld_plants.geojson")

## 3. Combined Overview Map

In [ ]:
print("Building grid overview map...")
m = folium.Map(location=[39.5, -98.35], zoom_start=4, tiles="CartoDB positron")

# Transmission lines — thin gray polylines
print("  Adding transmission lines...")
folium.GeoJson(
    tl_hv.__geo_interface__,
    name="Transmission Lines (≥200 kV)",
    style_function=lambda _: {
        "color": "#888888",
        "weight": 0.5,
        "opacity": 0.6,
    },
).add_to(m)

# Power plants — small red circle markers
print("  Adding HIFLD power plants...")
for _, row in hp.iterrows():
    if row.geometry is None:
        continue
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=2,
        color="red",
        fill=True,
        fill_opacity=0.6,
    ).add_to(m)

folium.LayerControl().add_to(m)

map_path = PROJECT_ROOT / "data" / "processed" / "grid_overview_map.html"
m.save(str(map_path))
print(f"Map saved → {map_path}")